In [ ]:
import os
import torch
from torchvision import transforms
from dataset import ShapesDataset 
from prototypical_net import ConvNet
from learn2learn.data import MetaDataset, TaskDataset
from learn2learn.data.transforms import NWays, KShots, LoadData, RemapLabels
import torch.nn.functional as F
import torch.nn as nn

In [ ]:
# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_root = "images/train-images-augmented"
class_names = [d for d in os.listdir(train_root) if os.path.isdir(os.path.join(train_root, d))]
n_ways = len(class_names)
k_shot = 5
k_query = 5
print(f"Classes: {class_names}")
print(f"N-way automatically set to: {n_ways}")

# Transform and dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])
dataset = ShapesDataset(train_root, transform=transform)
meta_dataset = MetaDataset(dataset)

# TaskSampler
taskset = TaskDataset(
    meta_dataset,
    task_transforms=[
        NWays(meta_dataset, n=n_ways),
        KShots(meta_dataset, k=k_shot + k_query),
        LoadData(meta_dataset),
        RemapLabels(meta_dataset),
    ],
    num_tasks=1000
)

# Model and optimizer
model = ConvNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

# Matching Networks Training Loop
for iteration in range(1000):
    try:
        task = taskset.sample()
        data, labels = task
        data, labels = data.to(device), labels.to(device)
        embeddings = model(data)

        support, support_labels = [], []
        query, query_labels = [], []

        for class_idx in range(n_ways):
            class_mask = labels == class_idx
            class_indices = torch.nonzero(class_mask).squeeze()

            if len(class_indices) < (k_shot + k_query):
                continue

            support_idx = class_indices[:k_shot]
            query_idx = class_indices[k_shot:k_shot + k_query]

            support.append(embeddings[support_idx])
            support_labels.append(labels[support_idx])
            query.append(embeddings[query_idx])
            query_labels.append(labels[query_idx])

        if len(support) < n_ways:
            print(f"Skipping task at iteration {iteration} due to insufficient samples.")
            continue

        support = torch.cat(support)
        support_labels = torch.cat(support_labels)
        query = torch.cat(query)
        query_labels = torch.cat(query_labels)

        # Normalize and compute cosine similarities
        support = F.normalize(support, dim=1)
        query = F.normalize(query, dim=1)
        sims = torch.mm(query, support.t())  # (n_query, n_support)

        # Soft nearest neighbor voting
        one_hot = torch.zeros(support.size(0), n_ways).to(device)
        one_hot.scatter_(1, support_labels.unsqueeze(1), 1)
        preds = torch.mm(F.softmax(sims, dim=1), one_hot)

        loss = loss_fn(preds, query_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        acc = (preds.argmax(1) == query_labels).float().mean()

        if iteration % 100 == 0:
            print(f"[MatchingNet] Iteration {iteration}: Loss={loss.item():.4f}, Accuracy={acc.item() * 100:.2f}%")

    except Exception as e:
        print(f"Error in iteration {iteration}: {e}")

# Save model
torch.save(model.state_dict(), "saved_models/matchingnet_model.pth")
print("✅ Matching Network model saved.")


Classes: ['apple', 'kiwi', 'rectangle']
N-way automatically set to: 3
[MatchingNet] Iteration 0: Loss=1.0130, Accuracy=100.00%
[MatchingNet] Iteration 100: Loss=0.8711, Accuracy=100.00%
[MatchingNet] Iteration 200: Loss=0.8711, Accuracy=100.00%
[MatchingNet] Iteration 300: Loss=0.8711, Accuracy=100.00%
[MatchingNet] Iteration 400: Loss=0.8712, Accuracy=100.00%
[MatchingNet] Iteration 500: Loss=0.8711, Accuracy=100.00%
[MatchingNet] Iteration 600: Loss=0.8711, Accuracy=100.00%
[MatchingNet] Iteration 700: Loss=0.8711, Accuracy=100.00%
[MatchingNet] Iteration 800: Loss=0.8711, Accuracy=100.00%
[MatchingNet] Iteration 900: Loss=0.8711, Accuracy=100.00%
✅ Matching Network model saved.
